# Digit Recognizer

In [ ]:
%load_ext tensorboard

Image classification refers to the task of assigning a label to an image. To test image classification and to test the effectiveness of different strategies a common dataset, MNIST, is usually applied because of its simplicity.

MNIST ("Modified National Institute of Standards and Technology") is the de facto “hello world” dataset of computer vision. Since its release in 1999, this classic dataset of handwritten images has served as the basis for benchmarking classification algorithms. As new machine learning techniques emerge, MNIST remains a reliable resource for researchers and learners alike.

In [ ]:
import numpy
import pandas
import matplotlib.pyplot as plt

from tqdm.notebook import trange

import torch
import torch.utils.tensorboard
import torchvision

import sklearn.metrics as metrics
from sklearn.model_selection import train_test_split

## Architecture

In deep learning, a convolutional neural network (CNN, or ConvNet) is a class of deep neural network, most commonly applied to analyze visual imagery. CNNs are regularized versions of multilayer perceptrons. Multilayer perceptrons usually mean fully connected networks, that is, each neuron in one layer is connected to all neurons in the next layer.

In [ ]:
class CNN(torch.nn.Module):
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()

        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(1, 64, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(inplace=True),
            torch.nn.Conv2d(64, 128, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(128),
            torch.nn.ReLU(inplace=True),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),
            torch.nn.Conv2d(128, 256, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(256),
            torch.nn.ReLU(inplace=True),
            torch.nn.Conv2d(256, 256, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(256),
            torch.nn.ReLU(inplace=True),
            torch.nn.MaxPool2d(kernel_size=2, stride=2)
        )

        for child in self.features.children():
            if isinstance(child, torch.nn.Conv2d):
                n = child.kernel_size[0] * child.kernel_size[1] * child.out_channels
                child.weight.data.normal_(0, numpy.sqrt(2.0 / n, dtype=float))
            elif isinstance(child, torch.nn.BatchNorm2d):
                child.weight.data.fill_(1)
                child.bias.data.zero_()

        self.classifier = torch.nn.Sequential(
            torch.nn.Dropout(0.5),
            torch.nn.Linear(256 * 7 * 7, 512),
            torch.nn.BatchNorm1d(512),
            torch.nn.ReLU(inplace=True),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(512, 1024),
            torch.nn.BatchNorm1d(1024),
            torch.nn.ReLU(inplace=True),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(1024, 10)
        )

        for child in self.classifier.children():
            if isinstance(child, torch.nn.Linear):
                torch.nn.init.xavier_uniform_(child.weight)
            elif isinstance(child, torch.nn.BatchNorm1d):
                child.weight.data.fill_(1)
                child.bias.data.zero_()

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)

        return self.classifier(x)

### GPU device availability

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

### Train properties

In [ ]:
batch_size = 64
epochs = 50

model = CNN().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=3.25e-3)
criterion = torch.nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.065)

## Loading data

The data files train.csv and test.csv contain gray-scale images of hand-drawn digits, from zero through nine.

Each image is 28 pixels in height and 28 pixels in width, for a total of 784 pixels in total. Each pixel has a single pixel-value associated with it, indicating the lightness or darkness of that pixel, with higher numbers meaning darker. This pixel-value is an integer between 0 and 255, inclusive.

The training data set, (train.csv), has 785 columns. The first column, called "label", is the digit that was drawn by the user. The rest of the columns contain the pixel-values of the associated image.

Each pixel column in the training set has a name like pixelx, where x is an integer between 0 and 783, inclusive. To locate this pixel on the image, suppose that we have decomposed x as x = i * 28 + j, where i and j are integers between 0 and 27, inclusive. Then pixelx is located on row i and column j of a 28 x 28 matrix, (indexing by zero).

For example, pixel31 indicates the pixel that is in the fourth column from the left, and the second row from the top, as in the ascii-diagram below.

Visually, if we omit the "pixel" prefix, the pixels make up the image like this:

```
000 001 002 003 ... 026 027
028 029 030 031 ... 054 055
056 057 058 059 ... 082 083
 |   |   |   |  ...  |   |
728 729 730 731 ... 754 755
756 757 758 759 ... 782 783
```

In [ ]:
train_csv = pandas.read_csv("../input/digit-recognizer/train.csv", dtype=numpy.uint8)

train_labels = train_csv.label.values
train_images = train_csv.loc[:, train_csv.columns != "label"].values / 255.0

images_train, images_eval, labels_train, labels_eval = train_test_split(
    train_images, train_labels, test_size = 0.2
)

images_train = torch.from_numpy(images_train).type(torch.FloatTensor)
labels_train = torch.from_numpy(labels_train).type(torch.LongTensor)

images_eval = torch.from_numpy(images_eval).type(torch.FloatTensor)
labels_eval = torch.from_numpy(labels_eval).type(torch.LongTensor)

train_dataset = torch.utils.data.TensorDataset(images_train, labels_train)
eval_dataset = torch.utils.data.TensorDataset(images_eval, labels_eval)

# Load test data
test_csv = pandas.read_csv("../input/digit-recognizer/test.csv")
test_images = train_csv.values
test_images = torch.from_numpy(test_images)
test_data = torch.utils.data.TensorDataset(test_images)
test = torch.utils.data.DataLoader(test_data, batch_size = batch_size, shuffle = True)

### Class distribution in datasets

In [ ]:
temp = [train_dataset[index][1] for index in range(len(train_dataset))]
values, count = numpy.unique(temp, return_counts=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5.2, 4.8), dpi=100)

fig.tight_layout()

temp = [train_dataset[index][1] for index in range(len(train_dataset))]
values, count = numpy.unique(temp, return_counts=True)

ax1.bar(values, count)
ax1.set_xlabel("Class")
ax1.set_ylabel("Count")
ax1.set_xticks(values)
ax1.set_title("Class distribution in train data")

temp = [eval_dataset[index][1] for index in range(len(eval_dataset))]
values, count = numpy.unique(temp, return_counts=True)

ax2.bar(values, count)
ax2.set_xlabel("Class")
ax2.set_ylabel("Count")
ax2.set_xticks(values)
ax2.set_title("Class distribution in evaluation data")

fig.subplots_adjust(hspace=0.5)

fig.show()

### Dataloader

In [ ]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
eval_loader = torch.utils.data.DataLoader(eval_dataset, batch_size = batch_size, shuffle = True)

## Train

In [ ]:
train_loss = []
train_accuracy = []
eval_loss = []
eval_accuracy = []

# Train intermediate variables
in_train_correct = 0
in_train_loss = 0
in_train_total = 0

# Evaluate intermediate variables
in_eval_correct = 0
in_eval_loss = 0
in_eval_total = 0

writer = torch.utils.tensorboard.SummaryWriter()

for index in trange(epochs, desc="Epoch"):
    with torch.enable_grad():
        model.train()
        for image, label in train_loader:
            image = torch.autograd.Variable(image.view(-1, 1, 28, 28)).to(device)
            label = torch.autograd.Variable(label).to(device)

            optimizer.zero_grad()

            output = model(image)
            loss = criterion(output, label)
            loss.backward()

            optimizer.step()

            in_train_loss += loss.item()
            in_train_total += len(label)

            _, prediction = output.data.max(1, keepdim=False)
            in_train_correct += int((prediction == label).sum())

    with torch.no_grad():
        model.eval()
        for image, label in eval_loader:
            image = torch.autograd.Variable(image.view(-1, 1, 28, 28)).to(device)
            label = torch.autograd.Variable(label).to(device)

            output = model(image)
            in_eval_loss += criterion(output, label).item()
            in_eval_total += len(label)

            _, prediction = output.data.max(1, keepdim=False)
            in_eval_correct += int((prediction == label).sum())

    scheduler.step()

    writer.add_scalar("Train/Accuracy", in_train_correct / in_train_total, index)
    writer.add_scalar("Train/Loss", in_train_loss / len(train_loader), index)
    writer.add_scalar("Evaluate/Accuracy", in_eval_correct / in_eval_total, index)
    writer.add_scalar("Evaluate/Loss", in_eval_loss / len(eval_loader), index)

writer.close()

## Evaluation

### Train/evaluate accuracy/loss

In [ ]:
%tensorboard --logdir runs

### Prediction on known evaluation data

In [ ]:
targets = torch.tensor([]).to(device)
predictions = torch.tensor([]).to(device)

with torch.no_grad():
    model.eval()
    for image, label in eval_loader:
        image = torch.autograd.Variable(image.view(-1, 1, 28, 28)).to(device)
        label = torch.autograd.Variable(label).to(device)

        output = model(image)

        targets = torch.cat((targets, label), dim=0)
        predictions = torch.cat((predictions, output), dim=0)

targets = numpy.array(targets.type(torch.LongTensor))

predictions = predictions.cpu()

In [ ]:
cm = metrics.confusion_matrix(targets, predictions.argmax(dim=1))

plt.figure(figsize=(10, 8.5))

plt.imshow(numpy.log2(cm + 1), cmap="Reds")
plt.colorbar()
plt.tick_params(size=5, color="white")
plt.xticks(numpy.arange(0, len(values)), numpy.arange(0, len(values)))
plt.yticks(numpy.arange(0, len(values)), numpy.arange(0, len(values)))

threshold = cm.max() / 2

for index in range(len(values)):
    for jndex in range(len(values)):
        if cm[index, jndex] > threshold:
            color = "w"
        else:
            color = "k"

        plt.text(jndex, index,
                 int(cm[index, jndex]),
                 horizontalalignment="center", color=color)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

### Error rate

In [ ]:
predictions = numpy.array(torch.max(predictions, 1)[1])
misclassified = numpy.unique(targets == predictions, return_counts=True)[1][0]

print(f"Error rate: {misclassified / len(predictions) * 100:.2f}%")
print(f"Score: {(len(predictions) - misclassified) / len(predictions) * 100:.2f}%")

## Make submission

In [ ]:
submission = pandas.read_csv("../input/digit-recognizer/test.csv", dtype=numpy.uint8).values / 255
submission = torch.from_numpy(submission).type(torch.FloatTensor)
submission = torch.utils.data.DataLoader(submission, batch_size=batch_size)

In [ ]:
with torch.no_grad():
    predictions = torch.tensor([]).to(device)

    for image in submission:
        image = torch.autograd.Variable(image.view(-1, 1, 28, 28)).to(device)
        output = model(image)

        predictions = torch.cat((predictions, output), dim=0)

In [ ]:
submission = []
for index, prediction in enumerate(predictions.argmax(dim=1)):
    submission.append([index + 1, prediction.cpu().numpy()])

In [ ]:
df = pandas.DataFrame(submission, columns=["ImageId", "Label"])
df.to_csv("submission.csv", index=False)

In [ ]:
print("Submission File Complete!")